In [6]:
import duckdb
import pandas as pd
from pathlib import Path

# Paths: notebook is in /notebooks, so go one level up
WAREHOUSE_PATH = Path("..") / "warehouse" / "covid_dw.duckdb"
PROC_DIR = Path("..") / "data" / "processed"

weekly = pd.read_csv(PROC_DIR / "covid_weekly_features.csv", parse_dates=["date"])

# Build dim_date
dim_date = (
    weekly[["date"]]
    .drop_duplicates()
    .sort_values("date")
    .assign(
        date_key=lambda d: d["date"].dt.strftime("%Y%m%d").astype(int),
        day=lambda d: d["date"].dt.day,
        month=lambda d: d["date"].dt.month,
        month_name=lambda d: d["date"].dt.month_name(),
        quarter=lambda d: d["date"].dt.quarter,
        year=lambda d: d["date"].dt.year,
        iso_week=lambda d: d["date"].dt.isocalendar().week.astype(int),
    )
)

# ✅ Reorder columns to match dim_date schema
dim_date = dim_date[[
    "date_key",
    "date",
    "day",
    "month",
    "month_name",
    "quarter",
    "year",
    "iso_week"
]]

# Build dim_country
dim_country = (
    weekly[["iso_code", "location", "continent", "population"]]
    .drop_duplicates()
    .rename(columns={"location": "country_name"})
    .reset_index(drop=True)
)
dim_country["country_key"] = dim_country.index + 1

# ✅ Reorder columns to match dim_country schema
dim_country = dim_country[[
    "country_key",
    "iso_code",
    "country_name",
    "continent",
    "population"
]]

# Merge keys into fact
fact = weekly.merge(dim_date[["date", "date_key"]], on="date", how="left")
fact = fact.merge(
    dim_country[["country_key", "iso_code"]],
    on="iso_code",
    how="left"
)

fact["fact_key"] = range(1, len(fact) + 1)

# IMPORTANT: no vaccination_rate here
fact_table = fact[[
    "fact_key", "date_key", "country_key",
    "new_cases", "new_deaths",
    "rolling_7d_cases", "rolling_7d_deaths",
    "cases_per_million", "deaths_per_million",
    "case_fatality_rate"
]]

con = duckdb.connect(str(WAREHOUSE_PATH))

# Register DataFrames as DuckDB views
con.register("df_dim_date", dim_date)
con.register("df_dim_country", dim_country)
con.register("df_fact", fact_table)

# Clear old data
con.execute("DELETE FROM dim_date")
con.execute("DELETE FROM dim_country")
con.execute("DELETE FROM fact_covid_metrics")

# Insert new data from registered views
con.execute("INSERT INTO dim_date SELECT * FROM df_dim_date")
con.execute("INSERT INTO dim_country SELECT * FROM df_dim_country")
con.execute("INSERT INTO fact_covid_metrics SELECT * FROM df_fact")

con.close()

print("Loaded dim_date, dim_country, fact_covid_metrics into DuckDB.")

Loaded dim_date, dim_country, fact_covid_metrics into DuckDB.
